<a href="https://colab.research.google.com/github/ZahraSCU/isba-2411-group-project-NLP/blob/main/Milestone2_Baseline_and_Representation_(Zahra).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/milestones/Milestone2_Baseline_and_Representation.ipynb)

> **Run this in Google Colab.** Click the badge above (or open the notebook from Canvas), then run the **Setup** cell first — it installs the dependencies. No local install needed.

# Assignment 2 · Milestone 2 — Project Baseline & Representation
**ISBA 2411 · Due Sunday, July 12, 2026 (11:59 PM PT) · Week 4**

Represent your team's own dataset and establish a first quantitative **baseline** for your problem.

## Deliverables
- **Completed notebook** — this notebook run on *your team's* data (demo data replaced at the swap-in cell).
- **500-word reflection** — in Section 5 of the notebook or a separate PDF.

## What to submit on Canvas
Submit **both** — the live work *and* an archival copy for the record:
1. **GitHub notebook link** — the URL to your team's completed notebook in your project repo (paste it in the *Website URL* box).
2. **PDF copy** — export the completed notebook to PDF (in Colab: *File → Print → Save as PDF*) and upload it (*File Upload*, PDF only).

> Canvas may only accept one submission method per attempt. Either way, **paste your GitHub notebook URL in the cell field below before exporting** so the link is also captured inside the PDF.

**Team GitHub notebook URL:** _(paste here before exporting to PDF)_

**Reading connections**

| Concept | Where to read |
|---|---|
| Tokens & embeddings | HOLLM Ch. 2; J&M Ch. 6 |
| Static vs. contextual representations | HOLLM Ch. 10; Tunstall Ch. 3 |
| Clustering / topic structure | HOLLM Ch. 5 |
| Semantic similarity / search | HOLLM Ch. 8 |

> The techniques below are the **default path**. If your project is classification, extraction, or summarization, adapt the baseline and metric to fit — keep the *represent → baseline → measure* structure.

> **How this notebook works.** It runs **end-to-end on a small built-in demo dataset** so you can see every step working before adapting it. Find the cell marked **`# ====== SWAP IN YOUR TEAM'S DATA HERE ======`** and replace the demo data with your own — everything downstream is written to work off the same variables.

> **Reproducibility.** A fixed random seed is set in the setup cell. On Colab, run the setup cell first (it installs dependencies).

### How this is graded
Scored on the shared milestone rubric ([`docs/ISBA2411_Assignment_Rubric.pdf`](../docs/ISBA2411_Assignment_Rubric.pdf)):

| Rubric criterion | In this milestone |
|---|---|
| **Execution** | Correct TF-IDF + contextual pipelines and a sound baseline on a real task |
| **Analysis & Insight** | Static-vs-contextual comparison + what the baseline number means |
| **Communication** | Clear reflection connecting results to the readings |


In [ ]:
# ===== Setup (Colab) =====
!pip install -q sentence-transformers scikit-learn pandas numpy

import numpy as np, pandas as pd, random
SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option('display.max_colwidth', 60)

## 0. Data
The demo dataset is a small set of customer-support tickets across three categories (`billing`, `technical`, `shipping`) with a `channel` field used later for a bias check.

**To use your own data:** replace the cell below so that you end up with a DataFrame `df` containing at least a `text` column (and, if you have them, a `label` column for classification and any group column for the bias check).

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/drive')

import os
os.chdir('/content/drive/Shared drives/ISBA 2411 Group 1/Data/')

from sklearn.model_selection import train_test_split

In [ ]:
DATA_PATH = "drugsComTrain_raw.xlsx"

raw = pd.read_excel(DATA_PATH)

# Drop rows with no condition, and a handful of rows with a stray HTML artifact in `condition`
raw = raw.dropna(subset=['condition', 'review', 'rating']).copy()
raw = raw[~raw['condition'].str.contains('</span>', na=False)]

def clean_text(t):
    t = str(t)
    t = t.replace('&#039;', "'").replace('&quot;', '"')
    t = t.strip('"').strip()
    return t

raw['text'] = raw['review'].apply(clean_text)

def sentiment_label(rating):
    if rating >= 7:
        return 'positive'
    if rating <= 4:
        return 'negative'
    return 'neutral'

raw['label'] = raw['rating'].apply(sentiment_label)

MENTAL_HEALTH_KEYWORDS = ['Depress', 'Anxiety', 'Bipolar', 'Panic', 'ADHD', 'Insomnia']
def group_label(condition):
    return 'mental_health' if any(k in condition for k in MENTAL_HEALTH_KEYWORDS) else 'other'

raw['group'] = raw['condition'].apply(group_label)

# Stratified sample by sentiment label, keeping the class balance of the full data


# N_SAMPLE is not defined, will define it for the snippet to run
N_SAMPLE = 50000
SEED = 42

raw['strata'] = raw['label'] + '_' + raw['group']
df, _ = train_test_split(raw, train_size=N_SAMPLE, random_state=SEED, stratify=raw['strata'])
df = df.reset_index(drop=True)

texts  = df['text'].tolist()
labels = df['label'].tolist()

print(len(df), 'documents |', df['label'].value_counts().to_dict())
print('group split:', df['group'].value_counts().to_dict())
df[['drugName', 'condition', 'text', 'rating', 'label', 'group']].head()

50000 documents | {'positive': 33150, 'negative': 12411, 'neutral': 4439}
group split: {'other': 39700, 'mental_health': 10300}


,drugName,condition,text,rating,label,group
0,Fentanyl,Pain,I too am extremely greatful for these Duragesi...,10,positive,other
1,Septra,Acne,This medicine has done only good things for me...,10,positive,other
2,Amiodarone,Ventricular Tachycardia,I had 3 t tests to stop my heart back to norma...,10,positive,other
3,Focalin XR,ADHD,Let me explain what can go wrong here and what...,9,positive,mental_health
4,Infliximab,Psoriatic Arthritis,I have been taking Remicade for several years ...,9,positive,other


## 1. Static representation (baseline)
TF-IDF — fast, interpretable, and the floor your contextual model must beat.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_static = tfidf.fit_transform(texts)
print('TF-IDF matrix:', X_static.shape, '(docs x vocab)')

TF-IDF matrix: (50000, 5000) (docs x vocab)


## 2. Contextual representation
Sentence-Transformers embeddings (`all-MiniLM-L6-v2`, ~90 MB) — dense vectors that capture meaning beyond surface words.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
X_ctx = embedder.encode(texts, show_progress_bar=False)
print('Contextual embeddings:', X_ctx.shape, '(docs x dim)')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Contextual embeddings: (50000, 384) (docs x dim)


## 3. A task with a metric
We run **both** a task-specific metric and an intrinsic check, on **both** representations:
- **Classification** (task metric): logistic regression, macro-F1 on a held-out split.
- **Clustering** (intrinsic check): KMeans + silhouette score.

Pick whichever is relevant to *your* project; the demo shows both.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, silhouette_score, classification_report
from sklearn.cluster import KMeans

def eval_classification(X, y, name):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    pred = clf.predict(Xte)
    clr = classification_report(yte, pred)

    result = {
        'rep': name,
        'accuracy': round(accuracy_score(yte, pred), 3),
        'macro_f1': round(f1_score(yte, pred, average='macro'), 3)
    }

    return result, clr

tfidf_result, tfidf_clr = eval_classification(X_static, labels, 'TF-IDF (static)')

minilm_result, minilm_clr = eval_classification(X_ctx, labels, 'MiniLM (contextual)')

clf_results = pd.DataFrame([tfidf_result, minilm_result])

print('Classification summary:')
print(clf_results.to_string(index=False))

print('\nTF-IDF classification report:')
print(tfidf_clr)

print('\nMiniLM classification report:')
print(minilm_clr)


Classification summary:
                rep  accuracy  macro_f1
    TF-IDF (static)     0.780     0.532
MiniLM (contextual)     0.749     0.480

TF-IDF classification report:
              precision    recall  f1-score   support

    negative       0.70      0.63      0.66      3723
     neutral       0.43      0.04      0.07      1332
    positive       0.81      0.94      0.87      9945

    accuracy                           0.78     15000
   macro avg       0.65      0.53      0.53     15000
weighted avg       0.75      0.78      0.74     15000


MiniLM classification report:
              precision    recall  f1-score   support

    negative       0.66      0.53      0.59      3723
     neutral       0.44      0.01      0.01      1332
    positive       0.77      0.93      0.84      9945

    accuracy                           0.75     15000
   macro avg       0.62      0.49      0.48     15000
weighted avg       0.71      0.75      0.71     15000



## 4. Quick bias check
Probe whether quality differs across a group in your data. The demo compares classification error across the `channel` group (`email` vs `chat`) — large gaps are a fairness red flag.

Adapt this to a group that matters for *your* domain (language, region, demographic proxy, …).

In [ ]:
idx = np.arange(len(df))
tr, te = train_test_split(idx, test_size=0.4, random_state=SEED, stratify=labels)
clf = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_ctx[tr], [labels[i] for i in tr])
rows = []
for g in df['group'].unique():
    mask = [i for i in te if df['group'].iloc[i] == g]
    if not mask:
        continue
    acc = accuracy_score([labels[i] for i in mask], clf.predict(X_ctx[mask]))
    rows.append(dict(group=g, n=len(mask), accuracy=round(acc, 3)))
print(pd.DataFrame(rows).to_string(index=False))
print('\nInterpretation: a large accuracy gap between groups suggests the representation or '
      'model serves one group better -- investigate before deploying.')

        group     n  accuracy
        other 15915     0.603
mental_health  4085     0.649

Interpretation: a large accuracy gap between groups suggests the representation or model serves one group better -- investigate before deploying.


## 5. Reflection (≈500 words — here or in a separate PDF)
- **Baseline.** What is your baseline number, and on what metric? Why is that the right metric for your problem?
- **Static vs. contextual.** What changed between TF-IDF and embeddings, and *why* (connect to HOLLM Ch. 2/10)?
- **Bias.** What did the group check show, and what would you do about a gap?
- **Next.** What will you try in Milestone 3 to *beat* this baseline?